# Lightweight Cytology-Aware Deep Representation and Shape-Guided Clustering
# for Automated Cervical Cell Detection from Pap-Smear Images

**End-to-end reference implementation** covering Phases 1–8 of the pipeline:

1. Data Collection
2. Image Preprocessing (Macenko normalization, Multi-scale Retinex, edge-preserving denoising)
3. Cell Region Identification (prompt-based candidate generation + boundary-aware supervision)
4. **Novelty 1** — Cytology-Aware Lightweight Representation (CALR)
5. Morphological Analysis (geometric features + Fourier shape descriptors)
6. **Novelty 2** — Shape-Guided Cytological Clustering (Adaptive Shape-Cytology Similarity, ASCS)
7. Automated Cell Detection (RT-DETRv2)
8. Cell Characterization (cytology-aware classifier)
9. Evaluation, efficiency benchmarking, and baseline comparisons

> **Notes before you run this**
> - This notebook is written to run on a GPU-enabled environment (Colab / Kaggle / local CUDA) with internet
>   access for package installation and dataset download. Cells that need network access are marked `# [NETWORK]`.
> - `EfficientViT`, `RT-DETRv2`, and `Segment-Anything (SAM)` are pulled from `timm` / `transformers`. Pin
>   versions as shown in the install cell to keep APIs stable.
> - Every module is runnable on synthetic/placeholder data out of the box (see the `USE_DUMMY_DATA` flag) so you
>   can sanity-check the pipeline before pointing it at the real Pap-smear dataset.
> - `TLM` and `CERVIA` baselines are cited comparison methods without a standard public reference
>   implementation; a clearly-marked stub is provided so you can drop in the official code/weights if you have
>   access to them. `ResNet-50` and `DenseNet-201` baselines are provided in full via `torchvision`.


## 0. Environment Setup

In [ ]:
# [NETWORK] Run once. Pin versions for reproducibility.
!pip -q install torch torchvision --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip -q install torch torchvision
!pip -q install timm==1.0.9 transformers==4.44.2 accelerate
!pip -q install opencv-python-headless scikit-image scikit-learn scipy pandas matplotlib seaborn
!pip -q install thop tqdm einops
!pip -q install segment-anything
!pip -q install opencv-contrib-python  # for guided filtering (cv2.ximgproc)


In [ ]:
import os, io, math, time, json, random, warnings, cmath
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from skimage import measure, filters, morphology, color
from scipy import ndimage as ndi
from scipy.fft import fft

from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              confusion_matrix, roc_curve, auc, matthews_corrcoef,
                              jaccard_score)
from sklearn.neighbors import NearestNeighbors

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Toggle this to False once the real dataset is downloaded and paths are set below.
USE_DUMMY_DATA = True


## Phase 1 — Data Collection

Source: Pap-smear dataset — https://zenodo.org/records/18427609

The helper below downloads and extracts the Zenodo record. Replace `RECORD_ID` / the file list if the record
layout differs; Zenodo records are static, so listing files first is the safest approach.

In [ ]:
# [NETWORK]
import requests, zipfile, pathlib

ZENODO_RECORD_ID = "18427609"
DATA_ROOT = pathlib.Path("./data/pap_smear")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def fetch_zenodo_record(record_id: str, out_dir: pathlib.Path):
    """Lists and downloads every file attached to a Zenodo record."""
    api_url = f"https://zenodo.org/api/records/{record_id}"
    meta = requests.get(api_url, timeout=30).json()
    files = meta.get("files", [])
    print(f"Found {len(files)} file(s) in record {record_id}")
    for f in files:
        fname = f["key"]
        link = f["links"]["self"]
        dest = out_dir / fname
        if dest.exists():
            print(f"  [skip] {fname} already present")
            continue
        print(f"  [downloading] {fname} ...")
        with requests.get(link, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(dest, "wb") as fh:
                for chunk in r.iter_content(chunk_size=1 << 20):
                    fh.write(chunk)
        if dest.suffix.lower() == ".zip":
            with zipfile.ZipFile(dest) as zf:
                zf.extractall(out_dir)
    return out_dir

if not USE_DUMMY_DATA:
    fetch_zenodo_record(ZENODO_RECORD_ID, DATA_ROOT)


In [ ]:
class PapSmearDataset(Dataset):
    """
    Expects a directory layout after extraction such as:
        DATA_ROOT/images/*.png
        DATA_ROOT/masks/cell/*.png      (optional, for Phase 3 supervision)
        DATA_ROOT/masks/nucleus/*.png
        DATA_ROOT/annotations.csv       (optional, bounding boxes / class labels for Phase 7-8)
    Adjust the globs below once you inspect the actual Zenodo record contents.
    """
    def __init__(self, root, split="train", transform=None, dummy_len=64, dummy_size=256):
        self.root = pathlib.Path(root)
        self.transform = transform
        self.dummy = USE_DUMMY_DATA
        self.dummy_len = dummy_len
        self.dummy_size = dummy_size
        if not self.dummy:
            self.image_paths = sorted((self.root / "images").glob("*.png"))
        else:
            self.image_paths = list(range(dummy_len))

    def __len__(self):
        return len(self.image_paths)

    def _synthetic_sample(self, size):
        """Generates a plausible-looking Pap-smear tile with a handful of stained blobs (cells)
        with darker nuclei, purely for pipeline sanity checks."""
        img = np.full((size, size, 3), 235, dtype=np.uint8)
        img = img - np.random.randint(0, 15, img.shape).astype(np.uint8)
        n_cells = np.random.randint(4, 10)
        cell_mask = np.zeros((size, size), np.uint8)
        nuc_mask = np.zeros((size, size), np.uint8)
        for _ in range(n_cells):
            cx, cy = np.random.randint(30, size - 30, 2)
            r_cyto = np.random.randint(18, 30)
            r_nuc = int(r_cyto * np.random.uniform(0.25, 0.55))
            cyto_color = (np.random.randint(150, 190), np.random.randint(120, 160), np.random.randint(170, 210))
            nuc_color = (np.random.randint(60, 100), np.random.randint(30, 60), np.random.randint(90, 130))
            cv2.circle(img, (cx, cy), r_cyto, cyto_color, -1, lineType=cv2.LINE_AA)
            cv2.circle(cell_mask, (cx, cy), r_cyto, 1, -1)
            cv2.circle(img, (cx, cy), r_nuc, nuc_color, -1, lineType=cv2.LINE_AA)
            cv2.circle(nuc_mask, (cx, cy), r_nuc, 1, -1)
        img = cv2.GaussianBlur(img, (3, 3), 0)
        noise = np.random.normal(0, 4, img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        boundary = cell_mask.astype(np.uint8) - cv2.erode(cell_mask, np.ones((3, 3), np.uint8))
        return img, cell_mask, nuc_mask, boundary

    def __getitem__(self, idx):
        if self.dummy:
            img, cell_mask, nuc_mask, boundary = self._synthetic_sample(self.dummy_size)
        else:
            p = self.image_paths[idx]
            img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            cell_mask = cv2.imread(str(self.root / "masks" / "cell" / p.name), 0)
            nuc_mask = cv2.imread(str(self.root / "masks" / "nucleus" / p.name), 0)
            boundary = (cell_mask.astype(np.uint8) - cv2.erode(cell_mask, np.ones((3, 3), np.uint8)))
        sample = {"image": img, "cell_mask": cell_mask, "nucleus_mask": nuc_mask, "boundary_map": boundary}
        if self.transform:
            sample = self.transform(sample)
        return sample

train_ds = PapSmearDataset(DATA_ROOT, split="train")
sample = train_ds[0]
print({k: (v.shape if hasattr(v, "shape") else v) for k, v in sample.items()})
plt.figure(figsize=(4, 4)); plt.imshow(sample["image"]); plt.title("Raw sample"); plt.axis("off"); plt.show()


## Phase 2 — Image Preprocessing

1. **Macenko stain normalization** — removes inter-slide/inter-scanner staining variation.
2. **Multi-scale Retinex (MSR)** — corrects uneven illumination.
3. **Edge-preserving denoising** — bilateral or guided filtering, preserves nuclear/cytoplasmic boundaries.

In [ ]:
class MacenkoNormalizer:
    """Stain separation & normalization following Macenko et al. (2009)."""
    def __init__(self, target_stain_matrix=None, alpha=1.0, beta=0.15):
        self.alpha = alpha
        self.beta = beta
        # Reference (target) stain vectors + max concentrations, commonly used defaults for H&E-like stains.
        self.target_stain_matrix = target_stain_matrix if target_stain_matrix is not None else np.array([
            [0.5626, 0.2159],
            [0.7201, 0.8012],
            [0.4062, 0.5581],
        ])
        self.target_max_c = np.array([1.9705, 1.0308])

    @staticmethod
    def _od(img):
        img = img.astype(np.float64)
        img[img == 0] = 1
        return -np.log(img / 255.0)

    def _estimate_stain_matrix(self, od, tissue_mask):
        od_hat = od[tissue_mask]
        if od_hat.shape[0] < 10:
            return self.target_stain_matrix, self.target_max_c
        cov = np.cov(od_hat.T)
        eigvals, eigvecs = np.linalg.eigh(cov)
        top2 = eigvecs[:, [-1, -2]]
        proj = od_hat @ top2
        angles = np.arctan2(proj[:, 1], proj[:, 0])
        min_a, max_a = np.percentile(angles, self.beta * 100), np.percentile(angles, (1 - self.beta) * 100)
        v1 = top2 @ np.array([np.cos(min_a), np.sin(min_a)])
        v2 = top2 @ np.array([np.cos(max_a), np.sin(max_a)])
        if v1[0] < v2[0]:
            stain_matrix = np.stack([v1, v2], axis=1)
        else:
            stain_matrix = np.stack([v2, v1], axis=1)
        stain_matrix = stain_matrix / np.linalg.norm(stain_matrix, axis=0, keepdims=True)
        concentrations = np.linalg.lstsq(stain_matrix, od_hat.T, rcond=None)[0]
        max_c = np.percentile(concentrations, 99, axis=1)
        return stain_matrix, max_c

    def normalize(self, image_rgb):
        h, w, _ = image_rgb.shape
        od = self._od(image_rgb).reshape(-1, 3)
        tissue_mask = (od.sum(axis=1) > self.beta)
        stain_matrix, max_c = self._estimate_stain_matrix(od, tissue_mask)
        concentrations = np.linalg.lstsq(stain_matrix, od.T, rcond=None)[0]
        norm_c = concentrations * (self.target_max_c / (max_c + 1e-8))[:, None]
        od_norm = self.target_stain_matrix @ norm_c
        img_norm = 255 * np.exp(-od_norm.T.reshape(h, w, 3))
        return np.clip(img_norm, 0, 255).astype(np.uint8)


def multi_scale_retinex(image_rgb, sigmas=(15, 80, 250), gain=1.0, offset=0.0):
    """MSR illumination correction, per-channel, then linear gain/offset + contrast stretch."""
    img = image_rgb.astype(np.float64) + 1.0
    msr = np.zeros_like(img)
    for sigma in sigmas:
        blurred = cv2.GaussianBlur(img, (0, 0), sigma) + 1.0
        msr += np.log(img) - np.log(blurred)
    msr /= len(sigmas)
    msr = gain * msr + offset
    out = np.zeros_like(msr)
    for c in range(3):
        ch = msr[..., c]
        lo, hi = np.percentile(ch, 1), np.percentile(ch, 99)
        out[..., c] = np.clip((ch - lo) / (hi - lo + 1e-8), 0, 1)
    return (out * 255).astype(np.uint8)


def edge_preserving_denoise(image_rgb, method="bilateral"):
    if method == "bilateral":
        return cv2.bilateralFilter(image_rgb, d=9, sigmaColor=50, sigmaSpace=50)
    elif method == "guided":
        try:
            guide = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
            return cv2.ximgproc.guidedFilter(guide, image_rgb, radius=8, eps=100)
        except AttributeError:
            print("cv2.ximgproc unavailable (install opencv-contrib-python) — falling back to bilateral filter.")
            return cv2.bilateralFilter(image_rgb, d=9, sigmaColor=50, sigmaSpace=50)
    raise ValueError(method)


def preprocess_pipeline(image_rgb, stain_normalizer, denoise_method="bilateral"):
    stain_norm = stain_normalizer.normalize(image_rgb)
    illum_corrected = multi_scale_retinex(stain_norm)
    denoised = edge_preserving_denoise(illum_corrected, method=denoise_method)
    return {"raw": image_rgb, "stain_normalized": stain_norm,
            "illumination_corrected": illum_corrected, "denoised": denoised}

macenko = MacenkoNormalizer()
stages = preprocess_pipeline(sample["image"], macenko)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (title, img) in zip(axes, stages.items()):
    ax.imshow(img); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## Phase 3 — Cell Region Identification

- **Foreground / cell-candidate generation**: a segmentation foundation model (SAM) used purely as a
  *lightweight, prompt-based candidate-region generator* — it proposes masks, it is not trained further.
- **Boundary-aware supervision**: a small, purpose-trained U-Net-style head predicts three co-registered maps
  (cell mask, nuclear mask, boundary map) supervised with a combined Dice + BCE + boundary loss, distilling the
  heavier foundation-model proposals into a compact, fast model used at inference time.

In [ ]:
# [NETWORK] Segment Anything as a *frozen* candidate-region generator.
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"   # download once: wget the official checkpoint URL
SAM_TYPE = "vit_b"

def load_sam_candidate_generator(checkpoint_path=SAM_CHECKPOINT, model_type=SAM_TYPE, device=DEVICE):
    if not os.path.exists(checkpoint_path):
        print(f"[warn] SAM checkpoint not found at {checkpoint_path}; "
              f"download from https://github.com/facebookresearch/segment-anything#model-checkpoints")
        return None
    sam = sam_model_registry[model_type](checkpoint=checkpoint_path).to(device)
    sam.eval()
    return SamAutomaticMaskGenerator(
        sam, points_per_side=24, pred_iou_thresh=0.86, stability_score_thresh=0.9,
        min_mask_region_area=60,
    )

def generate_cell_candidates(image_rgb, mask_generator):
    """Returns a list of binary candidate masks (foreground cell proposals). Falls back to classical
    Otsu + watershed candidate generation if SAM weights are unavailable, keeping the notebook runnable
    end-to-end without network access."""
    if mask_generator is not None:
        proposals = mask_generator.generate(image_rgb)
        return [p["segmentation"].astype(np.uint8) for p in proposals]
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    _, th = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = morphology.remove_small_objects(th.astype(bool), min_size=40)
    dist = ndi.distance_transform_edt(th)
    local_max = morphology.local_maxima(dist)
    markers = ndi.label(local_max)[0]
    labels = morphology.watershed(-dist, markers, mask=th) if hasattr(morphology, "watershed") \
        else __import__("skimage.segmentation", fromlist=["watershed"]).watershed(-dist, markers, mask=th)
    return [(labels == i).astype(np.uint8) for i in range(1, labels.max() + 1)]

sam_generator = load_sam_candidate_generator()
candidates = generate_cell_candidates(stages["denoised"], sam_generator)
print(f"{len(candidates)} cell candidates proposed")


In [ ]:
class BoundaryAwareUNet(nn.Module):
    """Compact U-Net predicting 3 co-registered maps: cell mask, nuclear mask, boundary map.
    Distills SAM's heavier proposals into a fast model usable at inference time."""
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        self.enc1 = block(in_ch, base)
        self.enc2 = block(base, base * 2)
        self.enc3 = block(base * 2, base * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = block(base * 4, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = block(base * 2, base)
        self.head = nn.Conv2d(base, 3, 1)  # [cell, nucleus, boundary]

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.head(d1))


def dice_loss(pred, target, eps=1e-6):
    pred, target = pred.flatten(1), target.flatten(1)
    inter = (pred * target).sum(1)
    return 1 - ((2 * inter + eps) / (pred.sum(1) + target.sum(1) + eps)).mean()


def boundary_aware_loss(pred, target, w_bce=1.0, w_dice=1.0, w_boundary=2.0):
    """pred/target: (B, 3, H, W) = [cell, nucleus, boundary]. Boundary channel gets extra weight since
    it is the thinnest / most imbalanced target and drives instance separability."""
    bce = F.binary_cross_entropy(pred, target, reduction="none")
    channel_w = torch.tensor([1.0, 1.0, w_boundary], device=pred.device).view(1, 3, 1, 1)
    bce = (bce * channel_w).mean()
    dsc = sum(dice_loss(pred[:, c], target[:, c]) for c in range(3)) / 3
    return w_bce * bce + w_dice * dsc, {"bce": bce.item(), "dice": dsc.item()}

region_model = BoundaryAwareUNet().to(DEVICE)
opt = torch.optim.AdamW(region_model.parameters(), lr=1e-3, weight_decay=1e-4)
print(region_model)


In [ ]:
def to_tensor_batch(sample, size=256):
    img = cv2.resize(sample["image"], (size, size)).astype(np.float32) / 255.0
    cell = cv2.resize(sample["cell_mask"], (size, size), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    nuc = cv2.resize(sample["nucleus_mask"], (size, size), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    bnd = cv2.resize(sample["boundary_map"], (size, size), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    x = torch.from_numpy(img).permute(2, 0, 1)
    y = torch.stack([torch.from_numpy(cell), torch.from_numpy(nuc), torch.from_numpy(bnd)], dim=0)
    return x, y

def train_region_model(model, dataset, epochs=3, batch_size=8, steps_per_epoch=20):
    model.train()
    for epoch in range(epochs):
        running = 0.0
        for step in range(steps_per_epoch):
            xs, ys = [], []
            for _ in range(batch_size):
                s = dataset[np.random.randint(len(dataset))]
                x, y = to_tensor_batch(s)
                xs.append(x); ys.append(y)
            x = torch.stack(xs).to(DEVICE)
            y = torch.stack(ys).to(DEVICE)
            opt.zero_grad()
            pred = model(x)
            loss, parts = boundary_aware_loss(pred, y)
            loss.backward()
            opt.step()
            running += loss.item()
        print(f"epoch {epoch+1}/{epochs}  loss={running/steps_per_epoch:.4f}")
    return model

region_model = train_region_model(region_model, train_ds, epochs=2, steps_per_epoch=8)


## Phase 4 — Novelty 1: Cytology-Aware Lightweight Representation (CALR)

- **EfficientViT backbone** (from `timm`) — lightweight multi-stage feature extractor.
- **Multi-scale morphology tokens** — mask-guided pooling of backbone feature maps at cellular, nuclear, and
  local-boundary scale, using the Phase-3 masks.
- **Nucleus–cytoplasm interaction** — cross-attention between nucleus tokens and cytoplasm tokens.
- **Cytology-guided feature gating** — an SE-style gate conditioned on external morphological features
  (Phase 5), so hand-crafted cytology priors modulate the deep channels.
- **Prototype-aware representation** — learnable class prototypes; final embedding is the fused representation
  concatenated with a similarity-to-prototype vector.

In [ ]:
import timm

class MaskGuidedPool(nn.Module):
    """Average-pools a feature map inside a (resized) binary mask -> one token per mask."""
    def forward(self, feat_map, mask):
        # feat_map: (B, C, H, W); mask: (B, 1, H, W) in [0, 1]
        mask = F.interpolate(mask, size=feat_map.shape[-2:], mode="bilinear", align_corners=False)
        num = (feat_map * mask).sum(dim=(2, 3))
        den = mask.sum(dim=(2, 3)).clamp(min=1e-6)
        return num / den  # (B, C)


class NucleusCytoplasmAttention(nn.Module):
    """Single-head cross-attention: nucleus token attends to cytoplasm token and vice-versa, then fuses."""
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.fuse = nn.Linear(dim * 2, dim)
        self.scale = dim ** -0.5

    def forward(self, nucleus_tok, cyto_tok):
        # tokens: (B, C) -> treat as length-1 sequences for cross attention
        q = self.q(nucleus_tok).unsqueeze(1)
        k = self.k(cyto_tok).unsqueeze(1)
        v = self.v(cyto_tok).unsqueeze(1)
        attn = torch.softmax((q @ k.transpose(-1, -2)) * self.scale, dim=-1)
        attended = (attn @ v).squeeze(1)
        return self.fuse(torch.cat([nucleus_tok, attended], dim=-1))


class CytologyGuidedGate(nn.Module):
    """SE-style channel gate, conditioned on external hand-crafted morphological features (Phase 5)."""
    def __init__(self, feat_dim, morph_dim=8, hidden=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(feat_dim + morph_dim, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, feat_dim), nn.Sigmoid())

    def forward(self, feat, morph_feats):
        gate = self.mlp(torch.cat([feat, morph_feats], dim=-1))
        return feat * gate


class PrototypeHead(nn.Module):
    """Learnable per-class prototypes; outputs similarity logits + the concatenated final embedding."""
    def __init__(self, feat_dim, num_classes, proj_dim=64):
        super().__init__()
        self.proj = nn.Linear(feat_dim, proj_dim)
        self.prototypes = nn.Parameter(torch.randn(num_classes, proj_dim) * 0.02)

    def forward(self, feat):
        z = F.normalize(self.proj(feat), dim=-1)
        protos = F.normalize(self.prototypes, dim=-1)
        sim = z @ protos.t()                       # (B, num_classes) similarity-to-prototype
        fused = torch.cat([feat, sim], dim=-1)      # prototype-aware representation
        return fused, sim


class CALR(nn.Module):
    """Cytology-Aware Lightweight Representation network (Novelty 1)."""
    def __init__(self, backbone_name="efficientvit_b0.r224_in1k", num_classes=2, morph_dim=8):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, features_only=True)
        chs = self.backbone.feature_info.channels()
        self.cell_ch, self.nuc_ch, self.bnd_ch = chs[-1], chs[-2], chs[-3]

        self.pool = MaskGuidedPool()
        # project the three scales to a common width before fusing
        common = 128
        self.proj_cell = nn.Linear(self.cell_ch, common)
        self.proj_nuc = nn.Linear(self.nuc_ch, common)
        self.proj_bnd = nn.Linear(self.bnd_ch, common)

        self.nc_attn = NucleusCytoplasmAttention(common)
        self.gate = CytologyGuidedGate(common * 2, morph_dim=morph_dim)  # cell+boundary concatenated width
        self.proto_head = PrototypeHead(common * 3, num_classes)

        fused_dim = common * 3 + num_classes
        self.classifier = nn.Sequential(nn.Linear(fused_dim, 128), nn.ReLU(inplace=True),
                                         nn.Dropout(0.2), nn.Linear(128, num_classes))

    def forward(self, image, cell_mask, nucleus_mask, boundary_mask, morph_feats):
        feats = self.backbone(image)                 # list of multi-stage feature maps
        f_bnd, f_nuc, f_cell = feats[-3], feats[-2], feats[-1]

        t_cell = self.proj_cell(self.pool(f_cell, cell_mask))       # cellular-scale token
        t_nuc = self.proj_nuc(self.pool(f_nuc, nucleus_mask))       # nuclear-scale token
        t_bnd = self.proj_bnd(self.pool(f_bnd, boundary_mask))      # local-boundary-scale token

        nc_fused = self.nc_attn(t_nuc, t_cell)                      # nucleus-cytoplasm interaction
        gated = self.gate(torch.cat([nc_fused, t_bnd], dim=-1), morph_feats)  # cytology-guided gating

        rep = torch.cat([gated, t_cell], dim=-1)
        fused, proto_sim = self.proto_head(rep)                     # prototype-aware representation
        logits = self.classifier(fused)
        return {"embedding": fused, "logits": logits, "prototype_similarity": proto_sim}


calr = CALR(num_classes=2, morph_dim=8).to(DEVICE)
n_params = sum(p.numel() for p in calr.parameters())
print(f"CALR parameters: {n_params/1e6:.2f}M")


## Phase 5 — Morphological Analysis

- **Geometric features**: area, perimeter, eccentricity, solidity, aspect ratio, circularity — via
  `skimage.measure.regionprops`.
- **Boundary features**: Fourier Shape Descriptors (rotation/scale/translation-normalized contour signature).

These feed both the CALR cytology-guided gate (Phase 4) and the shape embedding used in Phase 6 clustering.

In [ ]:
def geometric_features(mask):
    mask = (mask > 0).astype(np.uint8)
    props_list = measure.regionprops(measure.label(mask))
    if not props_list:
        return None
    p = max(props_list, key=lambda r: r.area)  # largest connected region = the cell
    area = p.area
    perimeter = p.perimeter if p.perimeter > 0 else 1.0
    eccentricity = p.eccentricity
    solidity = p.solidity
    aspect_ratio = (p.major_axis_length / p.minor_axis_length) if p.minor_axis_length > 0 else 1.0
    circularity = 4 * np.pi * area / (perimeter ** 2)
    return {"area": area, "perimeter": perimeter, "eccentricity": eccentricity,
            "solidity": solidity, "aspect_ratio": aspect_ratio, "circularity": circularity,
            "bbox": p.bbox, "centroid": p.centroid}


def fourier_shape_descriptors(mask, n_descriptors=10):
    """Rotation/translation/scale-normalized Fourier descriptors of the boundary contour."""
    mask = (mask > 0).astype(np.uint8)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.zeros(n_descriptors)
    contour = max(contours, key=cv2.contourArea).squeeze()
    if contour.ndim != 2 or contour.shape[0] < 8:
        return np.zeros(n_descriptors)
    complex_signature = contour[:, 0] + 1j * contour[:, 1]
    complex_signature -= complex_signature.mean()           # translation invariance
    descriptors = fft(complex_signature)
    mag = np.abs(descriptors)
    mag = mag / (mag[1] + 1e-8)                              # scale invariance (normalize by 1st harmonic)
    fd = mag[2:2 + n_descriptors]                             # drop DC (0) and 1st harmonic (rotation-sensitive)
    if len(fd) < n_descriptors:
        fd = np.pad(fd, (0, n_descriptors - len(fd)))
    return fd


def morphology_vector(cell_mask, geom_keys=("area", "perimeter", "eccentricity", "solidity",
                                             "aspect_ratio", "circularity")):
    """Fixed-length vector fed into CALR's cytology-guided gate (morph_dim=8 by default: 6 geometric + 2
    summary stats of the Fourier descriptors)."""
    geom = geometric_features(cell_mask)
    if geom is None:
        geom_vec = np.zeros(len(geom_keys))
    else:
        geom_vec = np.array([geom[k] for k in geom_keys], dtype=np.float32)
        geom_vec[0] = np.log1p(geom_vec[0])   # log-scale area/perimeter to tame magnitude
        geom_vec[1] = np.log1p(geom_vec[1])
    fd = fourier_shape_descriptors(cell_mask, n_descriptors=10)
    fd_summary = np.array([fd.mean(), fd.std()], dtype=np.float32)
    return np.concatenate([geom_vec, fd_summary]).astype(np.float32), fd

geom = geometric_features(candidates[0] if candidates else sample["cell_mask"])
mvec, fd = morphology_vector(candidates[0] if candidates else sample["cell_mask"])
print("Geometric features:", geom)
print("Morphology vector (-> CALR gate):", mvec)


## Phase 6 — Novelty 2: Shape-Guided Cytological Clustering

- **Deep cytological embedding**: the CALR `embedding` output (Phase 4).
- **Shape embedding**: normalized geometric features + Fourier descriptors (Phase 5).
- **Adaptive Shape-Cytology Similarity (ASCS)**: for each candidate pair, a learned/estimated gate decides how
  much weight appearance similarity vs. morphology similarity should receive, informed by local density.
- **Prototype-guided density clustering**: DBSCAN-style expansion, but using ASCS as the (precomputed) distance
  metric and prototype centers (from Phase 4) as seeds.
- **Shape consistency constraint**: candidate merges are rejected if they would push a cluster's internal shape
  variance above a threshold, discouraging heterogeneous morphology from being grouped together.

In [ ]:
def adaptive_shape_cytology_similarity(deep_emb, shape_emb, density, alpha_bounds=(0.2, 0.8)):
    """
    deep_emb, shape_emb: (N, D) L2-normalized embeddings.
    density: (N,) local density estimate (e.g. inverse mean k-NN distance in deep space), used to decide,
             per point, how much to trust deep-appearance vs. shape similarity: in dense, appearance-consistent
             regions favor deep similarity; in sparse regions favor the more robust shape cue.
    Returns: (N, N) ASCS similarity matrix in [0, 1].
    """
    deep_sim = deep_emb @ deep_emb.T                     # cosine sim, already normalized
    shape_dist = np.linalg.norm(shape_emb[:, None, :] - shape_emb[None, :, :], axis=-1)
    shape_sim = np.exp(-shape_dist / (shape_dist.std() + 1e-8))

    dens_norm = (density - density.min()) / (density.max() - density.min() + 1e-8)
    alpha_i = alpha_bounds[0] + (alpha_bounds[1] - alpha_bounds[0]) * dens_norm   # per-point deep-similarity weight
    alpha_pair = np.sqrt(np.outer(alpha_i, alpha_i))                              # symmetric pairwise weight

    ascs = alpha_pair * deep_sim + (1 - alpha_pair) * shape_sim
    return np.clip(ascs, 0, 1)


def local_density(deep_emb, k=5):
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(deep_emb))).fit(deep_emb)
    dist, _ = nn.kneighbors(deep_emb)
    mean_knn_dist = dist[:, 1:].mean(axis=1) + 1e-8
    return 1.0 / mean_knn_dist


def shape_consistency_ok(cluster_shape_feats, candidate_feat, max_std_ratio=1.5):
    """Rejects a merge if adding `candidate_feat` would push the cluster's per-dimension shape std beyond
    `max_std_ratio` times its current std (shape consistency constraint)."""
    if len(cluster_shape_feats) < 2:
        return True
    cur_std = np.std(cluster_shape_feats, axis=0) + 1e-6
    new_std = np.std(np.vstack([cluster_shape_feats, candidate_feat]), axis=0) + 1e-6
    return np.all(new_std <= max_std_ratio * cur_std)


def prototype_guided_density_clustering(deep_emb, shape_emb, prototypes=None, eps=0.55, min_pts=3,
                                         max_std_ratio=1.5):
    """DBSCAN-style clustering driven by the ASCS similarity matrix (converted to a distance) and, when
    prototype centers are supplied, seeded from the points closest to each prototype first."""
    n = len(deep_emb)
    density = local_density(deep_emb)
    ascs = adaptive_shape_cytology_similarity(deep_emb, shape_emb, density)
    dist = 1 - ascs
    np.fill_diagonal(dist, 0)

    labels = -np.ones(n, dtype=int)
    visited = np.zeros(n, dtype=bool)
    cluster_id = 0

    if prototypes is not None and len(prototypes) > 0:
        proto_sim = deep_emb @ np.asarray(prototypes).T
        seed_order = np.argsort(-proto_sim.max(axis=1))       # process prototype-like points first
    else:
        seed_order = np.argsort(-density)                      # else process densest points first

    for idx in seed_order:
        if visited[idx]:
            continue
        visited[idx] = True
        neighbors = np.where(dist[idx] <= eps)[0]
        if len(neighbors) < min_pts:
            labels[idx] = -1   # noise / singleton candidate
            continue
        labels[idx] = cluster_id
        cluster_shape_feats = [shape_emb[idx]]
        queue = list(neighbors)
        while queue:
            j = queue.pop()
            if visited[j]:
                if labels[j] == -1:
                    if shape_consistency_ok(np.array(cluster_shape_feats), shape_emb[j], max_std_ratio):
                        labels[j] = cluster_id
                        cluster_shape_feats.append(shape_emb[j])
                continue
            visited[j] = True
            if not shape_consistency_ok(np.array(cluster_shape_feats), shape_emb[j], max_std_ratio):
                labels[j] = -1
                continue
            labels[j] = cluster_id
            cluster_shape_feats.append(shape_emb[j])
            j_neighbors = np.where(dist[j] <= eps)[0]
            if len(j_neighbors) >= min_pts:
                queue.extend(list(j_neighbors))
        cluster_id += 1
    return labels, ascs


# --- Demo on synthetic candidate features ---
N = 30
rng = np.random.default_rng(0)
deep_emb_demo = rng.normal(size=(N, 32))
deep_emb_demo /= np.linalg.norm(deep_emb_demo, axis=1, keepdims=True)
shape_emb_demo = rng.normal(size=(N, 8))
labels, ascs = prototype_guided_density_clustering(deep_emb_demo, shape_emb_demo, eps=0.6, min_pts=2)
print("Cluster labels:", labels)
print("Num clusters (excl. noise):", len(set(labels)) - (1 if -1 in labels else 0))

plt.figure(figsize=(4, 4))
sns.heatmap(ascs, cmap="viridis"); plt.title("Adaptive Shape-Cytology Similarity (ASCS)")
plt.show()


## Phase 7 — Automated Cell Detection (RT-DETRv2)

RT-DETRv2 is loaded via `transformers` and fine-tuned on the cell-level bounding boxes derived from the
clustered/segmented candidates of Phases 3 & 6.

In [ ]:
# [NETWORK]
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor

RTDETR_CHECKPOINT = "PekingU/rtdetr_v2_r18vd"  # small/fast variant; swap for r34/r50/r101-vd for more capacity

def build_rtdetr(num_classes, checkpoint=RTDETR_CHECKPOINT, device=DEVICE):
    processor = RTDetrImageProcessor.from_pretrained(checkpoint)
    model = RTDetrV2ForObjectDetection.from_pretrained(
        checkpoint, num_labels=num_classes, ignore_mismatched_sizes=True
    ).to(device)
    return model, processor


def masks_to_boxes_and_labels(cell_masks, class_labels):
    """cell_masks: list of binary masks (from Phase 3/6). class_labels: parallel list of ints
    (e.g. 0=normal, 1=abnormal from Phase 8, or a finer Bethesda category)."""
    boxes, labels = [], []
    for m, cls in zip(cell_masks, class_labels):
        ys, xs = np.where(m > 0)
        if len(xs) == 0:
            continue
        x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
        boxes.append([x0, y0, x1, y1])
        labels.append(cls)
    return np.array(boxes, dtype=np.float32), np.array(labels, dtype=np.int64)


def rtdetr_train_step(model, processor, images, targets, optimizer):
    """images: list[np.ndarray HWC uint8]; targets: list[dict(boxes=(N,4) xyxy, class_labels=(N,))]."""
    coco_targets = []
    for t in targets:
        boxes_xywh = t["boxes"].copy()
        boxes_xywh[:, 2] -= boxes_xywh[:, 0]
        boxes_xywh[:, 3] -= boxes_xywh[:, 1]
        coco_targets.append({"boxes": torch.tensor(boxes_xywh), "class_labels": torch.tensor(t["class_labels"])})
    inputs = processor(images=images, annotations=None, return_tensors="pt").to(DEVICE)
    labels = [{"boxes": ct["boxes"].to(DEVICE), "class_labels": ct["class_labels"].to(DEVICE)} for ct in coco_targets]
    outputs = model(**inputs, labels=labels)
    loss = outputs.loss
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()

NUM_DETECTION_CLASSES = 2  # normal / abnormal cell, adjust to full cytology taxonomy if needed
det_model, det_processor = build_rtdetr(NUM_DETECTION_CLASSES)
det_optimizer = torch.optim.AdamW(det_model.parameters(), lr=1e-4, weight_decay=1e-4)
print("RT-DETRv2 loaded:", sum(p.numel() for p in det_model.parameters()) / 1e6, "M params")


In [ ]:
@torch.no_grad()
def rtdetr_predict(model, processor, image_rgb, score_thresh=0.5):
    model.eval()
    inputs = processor(images=image_rgb, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    target_sizes = torch.tensor([image_rgb.shape[:2]]).to(DEVICE)
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes,
                                                        threshold=score_thresh)[0]
    return {k: v.cpu().numpy() for k, v in results.items()}  # {'scores', 'labels', 'boxes'}

# quick synthetic fine-tuning loop (replace with a real DataLoader over the Zenodo dataset)
for step in range(3):
    imgs, tgts = [], []
    for _ in range(2):
        s = train_ds[np.random.randint(len(train_ds))]
        boxes, labels = masks_to_boxes_and_labels(candidates[:5] if candidates else [s["cell_mask"]],
                                                    [0] * min(5, max(1, len(candidates))))
        if len(boxes) == 0:
            continue
        imgs.append(s["image"]); tgts.append({"boxes": boxes, "class_labels": labels})
    if imgs:
        loss = rtdetr_train_step(det_model, det_processor, imgs, tgts, det_optimizer)
        print(f"step {step}: detection loss = {loss:.4f}")


## Phase 8 — Cell Characterization

A cytology-aware classification head reuses the CALR embedding (Phase 4) plus the RT-DETRv2 region-of-interest
crop to predict Normal / Abnormal (or a finer cytological category, e.g. ASC-US, LSIL, HSIL, per the Bethesda
system) for every detected cell.

In [ ]:
class CytologyClassifier(nn.Module):
    """Wraps CALR's embedding with a small MLP head; class taxonomy is configurable."""
    def __init__(self, calr_model, embed_dim, class_names=("normal", "abnormal")):
        super().__init__()
        self.calr = calr_model
        self.class_names = class_names
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, len(class_names)))

    def forward(self, image, cell_mask, nucleus_mask, boundary_mask, morph_feats):
        out = self.calr(image, cell_mask, nucleus_mask, boundary_mask, morph_feats)
        logits = self.head(out["embedding"])
        return {"logits": logits, "embedding": out["embedding"], "prototype_similarity": out["prototype_similarity"]}

CLASS_NAMES = ("normal", "abnormal")
embed_dim = 128 * 3 + len(CLASS_NAMES)  # matches CALR's fused_dim with num_classes=len(CLASS_NAMES)
calr_for_clf = CALR(num_classes=len(CLASS_NAMES), morph_dim=8).to(DEVICE)
cyto_classifier = CytologyClassifier(calr_for_clf, embed_dim, CLASS_NAMES).to(DEVICE)
print("Cytology classifier ready:", sum(p.numel() for p in cyto_classifier.parameters()) / 1e6, "M params")


## Phase 9 — Result & Discussion: Evaluation Metrics

### Detection metrics
Precision, Recall, F1, mAP, IoU.

### Segmentation / classification metrics
Dice, Jaccard/IoU, Pixel Accuracy, Accuracy, Precision, Recall, F1, Specificity, MCC.

### Diagnostics
Confusion Matrix, ROC Curve.

In [ ]:
def iou_boxes(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a; xb1, yb1, xb2, yb2 = box_b
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def detection_metrics(pred_boxes, pred_scores, pred_labels, gt_boxes, gt_labels, iou_thresh=0.5):
    """Greedy matching -> precision/recall/F1/mAP@iou_thresh, mean IoU of matched pairs."""
    order = np.argsort(-pred_scores)
    matched_gt = set()
    tp, fp = 0, 0
    ious = []
    for i in order:
        best_iou, best_j = 0, -1
        for j, gb in enumerate(gt_boxes):
            if j in matched_gt or pred_labels[i] != gt_labels[j]:
                continue
            iou = iou_boxes(pred_boxes[i], gb)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= iou_thresh:
            tp += 1; matched_gt.add(best_j); ious.append(best_iou)
        else:
            fp += 1
    fn = len(gt_boxes) - len(matched_gt)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    mean_iou = float(np.mean(ious)) if ious else 0.0
    # simplified single-threshold AP (area under a 1-point PR estimate); for full mAP, sweep multiple IoU
    # thresholds (e.g. 0.5:0.95:0.05 as in COCO) and average.
    ap = precision * recall
    return {"precision": precision, "recall": recall, "f1": f1, "AP": ap, "mean_IoU": mean_iou}


def segmentation_metrics(pred_mask, gt_mask):
    pred = (pred_mask > 0.5).astype(np.uint8).flatten()
    gt = (gt_mask > 0.5).astype(np.uint8).flatten()
    inter = (pred & gt).sum()
    dice = 2 * inter / (pred.sum() + gt.sum() + 1e-8)
    jaccard = jaccard_score(gt, pred, zero_division=0)
    pixel_acc = (pred == gt).mean()
    return {"dice": dice, "jaccard_iou": jaccard, "pixel_accuracy": pixel_acc}


def classification_metrics(y_true, y_pred, y_score=None, positive_label=1):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    specificity = tn / (tn + fp + 1e-8)
    mcc = matthews_corrcoef(y_true, y_pred)
    out = {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
           "specificity": specificity, "mcc": mcc, "confusion_matrix": cm}
    if y_score is not None:
        fpr, tpr, _ = roc_curve(y_true, y_score)
        out["roc_auc"] = auc(fpr, tpr)
        out["roc_curve"] = (fpr, tpr)
    return out


def plot_confusion_and_roc(cm, class_names, roc=None):
    fig, axes = plt.subplots(1, 2 if roc else 1, figsize=(10 if roc else 5, 4))
    ax0 = axes[0] if roc else axes
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax0)
    ax0.set_xlabel("Predicted"); ax0.set_ylabel("True"); ax0.set_title("Confusion Matrix")
    if roc:
        fpr, tpr = roc
        axes[1].plot(fpr, tpr, label=f"ROC curve")
        axes[1].plot([0, 1], [0, 1], "--", color="gray")
        axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
        axes[1].set_title("ROC Curve"); axes[1].legend()
    plt.tight_layout(); plt.show()

# --- Demo with synthetic predictions ---
y_true = np.random.randint(0, 2, 100)
y_score = np.clip(y_true + np.random.normal(0, 0.4, 100), 0, 1)
y_pred = (y_score > 0.5).astype(int)
clf_metrics = classification_metrics(y_true, y_pred, y_score)
print({k: v for k, v in clf_metrics.items() if k not in ("confusion_matrix", "roc_curve")})
plot_confusion_and_roc(clf_metrics["confusion_matrix"], CLASS_NAMES, clf_metrics["roc_curve"])


## Phase 9b — Computational Efficiency & Lightweight-Deployment Profiling

Parameters, FLOPs, on-disk model size, inference latency, and peak memory — the metrics that justify the
"lightweight" claims of CALR vs. the ResNet-50 / DenseNet-201 baselines.

In [ ]:
from thop import profile as thop_profile

def profile_model(model, dummy_inputs, name="model"):
    model.eval()
    n_params = sum(p.numel() for p in model.parameters())
    try:
        macs, _ = thop_profile(model, inputs=dummy_inputs, verbose=False)
        flops = macs * 2
    except Exception as e:
        flops = float("nan")
        print(f"[warn] FLOP counting failed for {name}: {e}")

    tmp_path = f"/tmp/{name}.pt"
    torch.save(model.state_dict(), tmp_path)
    size_mb = os.path.getsize(tmp_path) / (1024 ** 2)

    # latency
    with torch.no_grad():
        for _ in range(5):
            model(*dummy_inputs)  # warmup
        torch.cuda.synchronize() if DEVICE.type == "cuda" else None
        t0 = time.time()
        n_runs = 30
        for _ in range(n_runs):
            model(*dummy_inputs)
        torch.cuda.synchronize() if DEVICE.type == "cuda" else None
        latency_ms = (time.time() - t0) / n_runs * 1000

    mem_mb = None
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            model(*dummy_inputs)
        mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return {"model": name, "params_M": n_params / 1e6, "FLOPs_G": flops / 1e9,
            "size_MB": size_mb, "latency_ms": latency_ms, "peak_mem_MB": mem_mb}

dummy_img = torch.randn(1, 3, 224, 224).to(DEVICE)
dummy_mask = torch.rand(1, 1, 224, 224).to(DEVICE)
dummy_morph = torch.randn(1, 8).to(DEVICE)
calr_profile = profile_model(calr, (dummy_img, dummy_mask, dummy_mask, dummy_mask, dummy_morph), name="CALR")
print(calr_profile)


## Baselines for Comparison

- **ResNet-50** and **DenseNet-201** — provided in full via `torchvision`, fine-tuned on the same
  cell-crop classification task for a fair comparison against CALR.
- **TLM** and **CERVIA** — cited comparison methods without a standardized public reference implementation.
  A clearly-marked stub is provided; replace with the official released code/weights if you have access
  (or re-implement per the architecture described in the corresponding paper) so numbers are directly
  comparable rather than approximated.

In [ ]:
import torchvision.models as tvm

def build_resnet50_baseline(num_classes):
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m.to(DEVICE)

def build_densenet201_baseline(num_classes):
    m = tvm.densenet201(weights=tvm.DenseNet201_Weights.IMAGENET1K_V1)
    m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    return m.to(DEVICE)

class TLMBaselineStub(nn.Module):
    """Placeholder for the 'TLM' comparison method [17].
    Replace this stub with the authors' official implementation/weights, or a faithful re-implementation
    from the paper, before reporting head-to-head numbers."""
    def __init__(self, num_classes):
        super().__init__()
        raise NotImplementedError(
            "TLM has no standard public reference implementation bundled here. "
            "Plug in the official code/weights from the cited paper [17].")

class CERVIABaselineStub(nn.Module):
    """Placeholder for the 'CERVIA' comparison method [18]. Same caveat as TLMBaselineStub."""
    def __init__(self, num_classes):
        super().__init__()
        raise NotImplementedError(
            "CERVIA has no standard public reference implementation bundled here. "
            "Plug in the official code/weights from the cited paper [18].")

resnet50_baseline = build_resnet50_baseline(len(CLASS_NAMES))
densenet201_baseline = build_densenet201_baseline(len(CLASS_NAMES))
print("ResNet-50 params (M):", sum(p.numel() for p in resnet50_baseline.parameters()) / 1e6)
print("DenseNet-201 params (M):", sum(p.numel() for p in densenet201_baseline.parameters()) / 1e6)


## Consolidated Results Table

Populate this after running real evaluation on held-out data; the scaffold below shows the expected schema.

In [ ]:
results_schema = pd.DataFrame(columns=[
    "Model", "Precision", "Recall", "F1", "mAP", "IoU",
    "Dice", "Jaccard", "PixelAcc", "Accuracy", "Specificity", "MCC",
    "Params(M)", "FLOPs(G)", "Size(MB)", "Latency(ms)", "PeakMem(MB)"
])
for model_name in ["CALR (proposed)", "ResNet-50 [16]", "TLM [17]", "CERVIA [18]", "DenseNet-201 [19]"]:
    results_schema.loc[len(results_schema)] = [model_name] + [np.nan] * (len(results_schema.columns) - 1)
results_schema
